# 🎓 TP 1: Data Preprocessing & Preparation
* **Team Members:** Hassan Es-sebaiy & Abdellatif Assal
* **Master:** AICE (Artificial Intelligence & Computer Engineering)
* **Date:** October 2026
---

# Lab 1: Data Preprocessing and Preparation 📊
**Course:** Machine Learning   
**Topic:** Data Cleaning, Handling Outliers (IQR), Categorical Encoding, & Normalization  
**Dataset:** `warehouse_messy_data.csv`

---

## 🎯 Objectives
The main goal of this lab is to convert raw, messy data into a clean, structured dataset ready for Machine Learning algorithms:
1. Load dataset using **Pandas**.
2. Identify & handle missing values ($NaN$).
3. Remove redundant/unuseful columns and duplicates.
4. Detect and treat anomalies and outliers using the **Interquartile Range (IQR)** method.
5. Perform categorical variable encoding and numerical variable normalization.
6. Export the final clean dataset to **NumPy** array.

---

---
## 📦 Step 1: Import Libraries and Load Dataset

In this step, we prepare our environment by importing the essential data analysis libraries (`pandas` and `numpy`). Then, we load our raw dataset (`warehouse_messy_data.csv`) into a Pandas DataFrame to inspect its initial structure and columns.
---

In [51]:
import pandas as pd
import numpy as np


df = pd.read_csv('warehouse_messy_data.csv')


df.head()

,Product ID,Product Name,Category,Warehouse,Location,Quantity,Price,Supplier,Status,Last Restocked
0,1102,gadget y,ELECTRONICS,Warehouse 2,Aisle 1,300,9.99,Supplier C,In Stock,NaN
1,1435,gadget y,ELECTRONICS,Warehouse 2,Aisle 4,two hundred,19.99,Supplier C,Out of Stock,NaN
2,1860,widget a,CLOTHING,Warehouse 2,Aisle 3,100,19.99,Supplier B,In Stock,20/12/2022
3,1270,gadget z,TOYS,Warehouse 2,Aisle 4,50,49.99,Supplier B,In Stock,20/12/2022
4,1106,widget a,FURNITURE,Warehouse 3,Aisle 3,two hundred,9.99,Supplier D,Out of Stock,25/04/2023


In [52]:

print("Dataset Shape (Rows, Columns):", df.shape)

df.info()

Dataset Shape (Rows, Columns): (1000, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Product ID      1000 non-null   int64  
 1   Product Name    1000 non-null   object 
 2   Category        1000 non-null   object 
 3   Warehouse       1000 non-null   object 
 4   Location        1000 non-null   object 
 5   Quantity        842 non-null    object 
 6   Price           793 non-null    float64
 7   Supplier        1000 non-null   object 
 8   Status          1000 non-null   object 
 9   Last Restocked  800 non-null    object 
dtypes: float64(1), int64(1), object(8)
memory usage: 78.3+ KB


---
## 🔍 Step 2: Exploratory Data Analysis & Checking Missing Values

In this step, we inspect our dataset for missing values, duplicates, and data types to understand what needs to be cleaned.
---

In [53]:
# Check for missing values in each column
print("Missing values per column:")
print(df.isnull().sum())

# Check for duplicate rows
print("\nNumber of duplicate rows:", df.duplicated().sum())

Missing values per column:
Product ID          0
Product Name        0
Category            0
Warehouse           0
Location            0
Quantity          158
Price             207
Supplier            0
Status              0
Last Restocked    200
dtype: int64

Number of duplicate rows: 0


---
## 🧹 Step 3: Handling Duplicates and Missing Values

- **Duplicates Check:** Found 0 duplicate rows. The dataset contains unique records.
- **Missing Values Treatment:** We will remove or handle the rows containing missing values (`NaN`) in columns like `Quantity`, `Price`, and `Last Restocked`.
---

In [54]:
# 1. Supprimer les doublons (sécurité pour le pipeline)
df.drop_duplicates(inplace=True)
#2. Supprimer les lignes contenant des valeurs manquantes
df.dropna(subset=['Quantity', 'Price', 'Last Restocked'], inplace=True)

print("Missing values after cleaning:")
print(df.isnull().sum())

Missing values after cleaning:
Product ID        0
Product Name      0
Category          0
Warehouse         0
Location          0
Quantity          0
Price             0
Supplier          0
Status            0
Last Restocked    0
dtype: int64


---
## 📈 Step 4: Detecting and Handling Outliers (IQR Method)

In this step, we identify extreme values (outliers) in numerical columns (`Quantity`, `Price`) using the **Interquartile Range (IQR)** method and clip them to upper and lower bounds.
---

In [55]:
# Function to treat outliers using IQR capping (winsorization)
def handle_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Cap values outside the bounds
    df[column] = np.where(df[column] < lower_bound, lower_bound, 
                          np.where(df[column] > upper_bound, upper_bound, df[column]))



In [56]:
print(df[['Quantity', 'Price']].dtypes)
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')

Quantity     object
Price       float64
dtype: object


In [57]:
# Apply IQR handling to numerical columns
numerical_cols = ['Quantity', 'Price']
for col in numerical_cols:
    handle_outliers_iqr(df, col)

print("Outliers handled successfully using IQR bounds!")

Outliers handled successfully using IQR bounds!


---
## 🔢 Step 5: Categorical Encoding & Numerical Normalization

In this step, we prepare our features for machine learning algorithms:
1. **Categorical Encoding:** Convert text columns into numerical form using **One-Hot Encoding** (`pd.get_dummies`).
2. **Feature Normalization:** Scale numerical features (`Quantity`, `Price`) to a range between 0 and 1 using **MinMaxScaler**.
---

In [58]:
from sklearn.preprocessing import MinMaxScaler


categorical_cols = ['Category', 'Warehouse', 'Status']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)


scaler = MinMaxScaler()
numerical_cols = ['Quantity', 'Price']
df_encoded[numerical_cols] = scaler.fit_transform(df_encoded[numerical_cols])


df_encoded.head()

,Product ID,Product Name,Location,Quantity,Price,Supplier,Last Restocked,Category_ELECTRONICS,Category_FURNITURE,Category_TOYS,Warehouse_Warehouse 2,Warehouse_Warehouse 3,Status_Low Stock,Status_Out of Stock
2,1860,widget a,Aisle 3,0.285714,0.25,Supplier B,20/12/2022,False,False,False,True,False,False,False
3,1270,gadget z,Aisle 4,0.000000,1.00,Supplier B,20/12/2022,False,False,True,True,False,False,False
4,1106,widget a,Aisle 3,NaN,0.00,Supplier D,25/04/2023,False,True,False,False,True,False,True
6,1700,widget a,Aisle 2,NaN,1.00,Supplier B,20/12/2022,False,False,False,True,False,False,False
7,1020,widget c,Aisle 5,NaN,0.00,Supplier D,20/12/2022,False,False,False,False,False,False,True


---
## 🏁 Step 6: Conversion to NumPy Array

In this final step, we convert our processed Pandas DataFrame into a NumPy array, which is the standard input format required by Scikit-Learn machine learning models.
---

In [59]:
# Convert the encoded and cleaned DataFrame to a NumPy array
X_array = df_encoded.to_numpy()

# Check the type and shape of the resulting array
print("Type of X_array:", type(X_array))
print("Shape of X_array:", X_array.shape)

# Display the first 3 rows of the array
print("\nFirst 3 rows of the NumPy array:")
print(X_array[:3])

Type of X_array: <class 'numpy.ndarray'>
Shape of X_array: (524, 14)

First 3 rows of the NumPy array:
[[1860 ' widget a ' 'Aisle 3' 0.2857142857142857 0.24999999999999994
  'Supplier B' '20/12/2022' False False False True False False False]
 [1270 ' gadget z ' 'Aisle 4' 0.0 1.0 'Supplier B' '20/12/2022' False
  False True True False False False]
 [1106 ' widget a ' 'Aisle 3' nan 0.0 'Supplier D' '25/04/2023' False
  True False False True False True]]
